In [1]:
import pandas as pd
import os
import glob
import tkinter as tk
from tkinter import filedialog

In [2]:
# ダイアログの裏回りを防ぐための設定
root = tk.Tk()
root.withdraw() # 余分なメインウィンドウを隠す
root.attributes('-topmost', True) # ダイアログを常に最前面に表示する

# フォルダ選択ダイアログを表示
print("フォルダ選択ダイアログを開いています...")
target_dir = filedialog.askdirectory(title="CSVファイルが入っているメインフォルダを選択してください")

if not target_dir:
    print("キャンセルされました。対象フォルダが選択されていません。")
else:
    print(f"選択されたフォルダ:\n{target_dir}\n")
    print("次のセルを実行して、処理を開始してください。")

フォルダ選択ダイアログを開いています...
選択されたフォルダ:
C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/学習用データ/10shot分/樹脂温度

次のセルを実行して、処理を開始してください。


In [3]:
if not target_dir:
    print("セル2でフォルダが選択されていません。もう一度セル2を実行してください。")
else:
    # 選択したフォルダ内のすべてのCSVファイルパスを取得
    all_csv_files = glob.glob(os.path.join(target_dir, "*.csv"))
    
    # すでに出力済みの集計結果ファイルが再度処理されるのを防ぐ
    target_files = [f for f in all_csv_files if "_10行ごと集計結果" not in f]
    
    if not target_files:
        print("指定されたフォルダ内に処理対象のCSVファイルが見つかりませんでした。")
    else:
        print(f"合計 {len(target_files)} 件のファイルを処理します。\n" + "-"*30)
        
        for file_path in target_files:
            filename = os.path.basename(file_path)
            print(f"処理中: {filename} ...")
            
            try:
                # CSVファイルの読み込み
                df = pd.read_csv(file_path, encoding='utf-8')

                # 最初の4列（そのまま1行目の値を抽出する列）と、それ以降の列（計算する列）を分ける
                first_4_cols = df.columns[:4].tolist()
                calc_cols = df.columns[4:].tolist()

                # 10行ごとにグループ化
                group_ids = df.index // 10

                # 最初の4列は、各グループの最初の行（1行目）の値を取得
                df_first = df[first_4_cols].groupby(group_ids).first()

                # 残りの列（5列目以降）は、平均と標準偏差を計算
                df_calc = df[calc_cols]
                df_mean = df_calc.groupby(group_ids).mean()
                df_std = df_calc.groupby(group_ids).std().fillna(0)

                # 新しい列名リストを作成
                new_cols = first_4_cols.copy()
                for col in calc_cols:
                    new_cols.extend([f'{col}_平均', f'{col}_標準偏差上', f'{col}_標準偏差下'])

                # 結果を格納するデータフレームを作成し、値を代入
                df_result = pd.DataFrame(index=df_mean.index, columns=new_cols)
                
                for col in first_4_cols:
                    df_result[col] = df_first[col]
                    
                for col in calc_cols:
                    mean_series = df_mean[col]
                    std_series = df_std[col]
                    df_result[f'{col}_平均'] = mean_series
                    df_result[f'{col}_標準偏差上'] = mean_series + std_series
                    df_result[f'{col}_標準偏差下'] = mean_series - std_series

                # 出力ファイルパスの作成 (同じフォルダに保存)
                output_filename = filename.replace('.csv', '_10行ごと集計結果.csv')
                output_path = os.path.join(target_dir, output_filename)
                
                # CSV書き出し
                df_result.to_csv(output_path, encoding='utf-8-sig', index=False)
                print(f"  → 出力完了: {output_filename}\n")
                
            except Exception as e:
                # 何らかのエラーが起きた場合はスキップして次のファイルへ
                print(f"  × エラーが発生しました: {filename} (理由: {e})\n")

        print("-" * 30 + "\nすべてのファイルの処理が完了しました！")

合計 3 件のファイルを処理します。
------------------------------
処理中: 最高値換算結果_summary_all_features_列入れ替え削除(手作業).csv ...
  → 出力完了: 最高値換算結果_summary_all_features_列入れ替え削除(手作業)_10行ごと集計結果.csv

処理中: 相対値換算結果_summary_all_features_列入れ替え削除.csv ...
  → 出力完了: 相対値換算結果_summary_all_features_列入れ替え削除_10行ごと集計結果.csv

処理中: 複合換算結果_summary_all_features.csv ...
  → 出力完了: 複合換算結果_summary_all_features_10行ごと集計結果.csv

------------------------------
すべてのファイルの処理が完了しました！
